# Os passes de cada seleção em cada Copa

## Task 3: calcular os passes de cada seleção em cada Copa e a média por partida

## O problema

Duas etapas: os passes de cada seleção **em cada partida**; depois, a soma **por Copa**, com o número de partidas e a
média.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-shuffle")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

A célula abaixo já vem pronta: o código dos vídeos 4.1 e 4.2 (lances, partidas e o ano em cada lance).

In [ ]:
# vídeo 4.1: os lances e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

# vídeo 4.2: o ano em cada lance
lances = eventos.join(partidas.select("match_id", "ano"), on="match_id")

## Passes de cada seleção em cada partida

## O que é um shuffle

Para contar, as linhas de cada grupo precisam se encontrar numa partição: o **shuffle**.

Procure no plano os dois `HashAggregate` e o `Exchange` entre eles: o que cada um faz?

| Tipo | Precisa de dado de outras partições? | Exemplos |
|---|---|---|
| **estreita** | não | `filter`, `select`, `withColumn` |
| **larga** | sim, exige shuffle | `groupBy`, `join` (sem broadcast), `distinct`, `orderBy`, `dropDuplicates` |

Cada shuffle divide o trabalho em **stages** (etapas).

## Agora por Copa

## Quantos shuffles?

Quantos `Exchange` o plano tem? De qual linha do código vem cada um?

## O número 200

`spark.sql.shuffle.partitions`: o número de partições depois de um shuffle (padrão: **200**). O **AQE** (Adaptive
Query Execution, o `AdaptiveSparkPlan` do topo do plano) olha o tamanho real do dado durante a execução e **junta as
partições pequenas**.

Na Spark UI, aba **SQL / DataFrame**, o ajuste do AQE aparece como **`AQEShuffleRead`** (vídeo 4.4).

## Para consultar depois

- **`repartition(n)`** redistribui o dado em `n` partições **com shuffle**, equilibradas; **`coalesce(n)`** só junta
  partições vizinhas, **sem shuffle**, e só diminui o número. O `repartition("coluna")` volta na aula 5, na gravação.
- **`orderBy`** também é uma transformação larga: para ordenar o dado inteiro, cada faixa de valores vai para uma
  partição (`rangepartitioning` no plano).
- **Mudando o padrão:** `spark.conf.set("spark.sql.shuffle.partitions", "8")`. Com o AQE ligado, raramente é
  necessário.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()